In [ ]:
# ===== Mine rules + stats from TRAIN ATTACKS ONLY (3,116 rows) -> rules_attack.json =====
import json, numpy as np, pandas as pd

tr  = pd.read_csv('/content/train.csv')
att = tr[tr['is_attack'] == True].drop(columns=['is_attack']).reset_index(drop=True)
R   = json.load(open('/content/rules.json'))            # benign rules: used only for formula list / column order
print('train attacks:', len(att))
num = att.apply(pd.to_numeric, errors='coerce')

# ---------- 1. Constants (single value) and always-empty columns ----------
pinned, always_empty = {}, []
for c in att.columns:
    s = att[c]
    if s.isna().all(): always_empty.append(c); continue
    if s.dropna().nunique() == 1 and s.notna().all():
        v = s.dropna().iloc[0]
        pinned[c] = v.item() if hasattr(v, 'item') else v
print('\n===== CONSTANTS in attack =====')
for c, v in pinned.items(): print(f'  {c:28s} = {v}')
print('\n===== ALWAYS EMPTY in attack =====\n ', always_empty)

ben_pin = R['pinned']
print('\n===== Benign pinned vs attack =====')
for c, v in ben_pin.items():
    print(f'  {c:24s} benign={v!s:10s} attack={pinned.get(c, "NOT constant")}')

# ---------- 2. Verify derived formulas on attack ----------
print('\n===== Derived formulas on attack (% rows matching, NaN-safe) =====')
deriv_ok = {}
for col, expr in R['derived'].items():
    try:
        pred = pd.eval(expr, local_dict={c: num[c] for c in num.columns}, engine='python')
        pred = pd.Series(pred, index=num.index).replace([np.inf, -np.inf], np.nan)
        act  = num[col]
        both_na = pred.isna() & act.isna()
        close   = np.isclose(pred.fillna(-1e18), act.fillna(-1e18), rtol=1e-3, atol=1e-6)
        rate = float((both_na | close).mean())
    except Exception as e:
        rate = float('nan')
    deriv_ok[col] = rate
    flag = '' if rate > 0.99 else '   <-- CHECK'
    print(f'  {col:36s} {rate:6.1%}{flag}')

# ---------- 3. Empty-value rules ----------
pk  = num['pkts_fwd']; dur = num['duration']
GROUPS = {
  'pps_bps': ['pps_fwd','pps_bwd','pps_total','bps_total','bps_fwd','bps_bwd'],
  'INT':     ['mean_hop_count','mean_hop_latency','latency_min','latency_max','latency_std',
              'queue_occupancy_mean','queue_occupancy_std','queue_occupancy_max'],
  'IAT':     ['mean_iat_int_ns','iat_std_int_ns','iat_min_int_ns','iat_max_int_ns'],
}
empty_rules = {}
print('\n===== Empty-value rules on attack =====')
for g, cols in GROUPS.items():
    na = att[cols].isna()
    any_na, all_na = na.any(axis=1), na.all(axis=1)
    r = {
      'n_empty_rows':          int(any_na.sum()),
      'all_or_none':           float((all_na | ~any_na).mean()),
      'empty_when_pkts1':      float(any_na[pk == 1].mean()) if (pk == 1).any() else None,
      'empty_when_pkts2':      float(any_na[pk == 2].mean()) if (pk == 2).any() else None,
      'filled_when_pkts_ge3':  float((~any_na[pk >= 3]).mean()) if (pk >= 3).any() else None,
      'empty_iff_duration0':   float((any_na == (dur == 0)).mean()),
    }
    empty_rules[g] = r
    print(f'  {g}: ' + ', '.join(f'{k}={v:.1%}' if isinstance(v, float) else f'{k}={v}' for k, v in r.items()))

print('\nduration==0 <=> pkts_fwd==1:', f'{((dur == 0) == (pk == 1)).mean():.1%}')
print('mean_hop_count==1 when present:', f'{(num["mean_hop_count"].dropna() == 1).mean():.1%}')

# ---------- 4. Stats for Section 2 (same format as the benign prompt) ----------
FIELDS = [c for c in R['core_columns'] if c not in ('timeframe_id','proto','ip_a','port_a','port_b')
          and c not in pinned and c not in always_empty]
def fmt(v):
    if pd.isna(v): return 'NA'
    v = float(v); return str(int(v)) if v == int(v) else f'{v:.4g}'
lines, stats = [], {}
for c in FIELDS:
    s = num[c]; x = s.dropna()
    q = x.quantile([.05, .25, .5, .75, .95]).tolist()
    zr, nr = float((x == 0).mean()) if len(x) else 0.0, float(s.isna().mean())
    stats[c] = dict(p5=q[0], p25=q[1], p50=q[2], p75=q[3], p95=q[4], min=float(x.min()),
                    max=float(x.max()), zero_rate=round(zr, 3), empty_rate=round(nr, 3))
    line = f'- {c}: typical {fmt(q[1])} to {fmt(q[3])}, middle {fmt(q[2])}, range {fmt(q[0])} to {fmt(q[4])} (max {fmt(x.max())})'
    extra = ([f'0 in {zr:.0%}'] if zr >= .01 else []) + ([f'NA in {nr:.0%}'] if nr >= .01 else [])
    lines.append(line + ('; ' + ', '.join(extra) if extra else ''))
ATTACK_STATS = '\n'.join(lines)
open('/content/attack_stats_section.txt', 'w').write(ATTACK_STATS)
print('\n===== Section 2 text (attack stats) =====\n' + ATTACK_STATS)

# ---------- 5. Shape facts + ids ----------
print(f'\nsingle-packet flows: {(pk == 1).mean():.1%}')
print('port_a top:', att['port_a'].value_counts().head(5).to_dict(), '| unique:', att['port_a'].nunique())
print('port_b top:', att['port_b'].value_counts().head(5).to_dict(), '| unique:', att['port_b'].nunique())
print('flows per timeframe:', att['timeframe_id'].value_counts().sort_index().to_dict())

# ---------- 6. Save ----------
out = {'source': 'train.csv attacks only', 'n_rows': len(att), 'pinned': pinned,
       'always_empty': always_empty, 'derived': R['derived'], 'derived_match_rate': deriv_ok,
       'empty_rules': empty_rules, 'attack_stats': stats,
       'id_formats': {'port_a_top': att['port_a'].value_counts().head(10).astype(int).to_dict(),
                      'port_b_top': att['port_b'].value_counts().head(10).astype(int).to_dict(),
                      'timeframes': sorted(att['timeframe_id'].unique().tolist())}}
json.dump(out, open('/content/rules_attack.json', 'w'), indent=1, default=str)
print('\nSaved: /content/rules_attack.json, /content/attack_stats_section.txt')

In [ ]:
# ===== Setting B: mine prompt rules + statistics from TRAIN ATTACK flows only =====
# Same checks as the benign mining (Setting A), different input data.
import json, os, re
import numpy as np
import pandas as pd

TRAIN  = '/content/train.csv'
RULES  = '/content/rules.json'
PROMPT = '/content/prompt_attack.txt'
OUTDIR = '/content'

tr  = pd.read_csv(TRAIN)
att = tr[tr['is_attack'] == True].drop(columns=['is_attack']).reset_index(drop=True)
R   = json.load(open(RULES))
num = att.apply(pd.to_numeric, errors='coerce')
print(f'train rows: {len(tr)} | attack rows used: {len(att)} (test data NOT used)')

# ---------- 1. Constant fields and always-empty fields ----------
pinned, always_empty = {}, []
for c in att.columns:
    s = att[c]
    if s.isna().all():
        always_empty.append(c); continue
    if s.notna().all() and s.nunique() == 1:
        v = s.iloc[0]
        pinned[c] = v.item() if hasattr(v, 'item') else v
print('\n===== 1. CONSTANT fields in attack =====')
for c, v in pinned.items(): print(f'  {c:34s} = {v}')
print('\n===== 2. ALWAYS-EMPTY fields in attack =====\n ', always_empty)

# ---------- 3. Derived formulas (from benign rules) checked on attack ----------
print('\n===== 3. Derived formulas on attack (% rows matching) =====')
deriv_ok = {}
for col, expr in R['derived'].items():
    try:
        pred = pd.eval(expr, local_dict={c: num[c] for c in num.columns}, engine='python')
        pred = pd.Series(pred, index=num.index).replace([np.inf, -np.inf], np.nan)
        act  = num[col]
        ok   = (pred.isna() & act.isna()) | np.isclose(pred.fillna(-1e18), act.fillna(-1e18), rtol=1e-3, atol=1e-6)
        rate_ = float(ok.mean())
    except Exception:
        rate_ = float('nan')
    deriv_ok[col] = rate_
    print(f'  {col:36s} {rate_:6.1%}' + ('' if rate_ > 0.99 else '   <-- CHECK'))

# ---------- 4. Empty-value rules ----------
pk, dur = num['pkts_fwd'], num['duration']
GROUPS = {
    'pps_bps': ['pps_fwd', 'pps_bwd', 'pps_total', 'bps_total', 'bps_fwd', 'bps_bwd'],
    'INT':     ['mean_hop_count', 'mean_hop_latency', 'latency_min', 'latency_max', 'latency_std',
                'queue_occupancy_mean', 'queue_occupancy_std', 'queue_occupancy_max'],
    'IAT':     ['mean_iat_int_ns', 'iat_std_int_ns', 'iat_min_int_ns', 'iat_max_int_ns'],
}
def rate(mask): return float(mask.mean()) if len(mask) else None
empty_rules = {}
print('\n===== 4. Empty-value rules on attack =====')
for g, cols in GROUPS.items():
    na = att[cols].isna()
    any_na, all_na = na.any(axis=1), na.all(axis=1)
    r = {'n_empty_rows':         int(any_na.sum()),
         'all_or_none':          rate(all_na | ~any_na),
         'empty_when_pkts1':     rate(any_na[pk == 1]),
         'empty_when_pkts2':     rate(any_na[pk == 2]),
         'filled_when_pkts_ge3': rate(~any_na[pk >= 3]),
         'empty_iff_duration0':  rate(any_na == (dur == 0))}
    empty_rules[g] = r
    print(f'  {g:8s}: ' + ', '.join(f'{k}={v:.1%}' if isinstance(v, float) else f'{k}={v}' for k, v in r.items()))
print(f'  duration==0 <=> pkts_fwd==1   : {rate((dur == 0) == (pk == 1)):.1%}')
print(f'  mean_hop_count==1 when present: {rate(num["mean_hop_count"].dropna() == 1):.1%}')

# ---------- 5. Per-field statistics (same 44 fields + format as benign Section 2) ----------
SKIP   = ('timeframe_id', 'proto', 'ip_a', 'port_a', 'port_b')
FIELDS = [c for c in R['core_columns'] if c not in SKIP]
def fmt(v):
    if pd.isna(v): return 'NA'
    v = float(v)
    return str(int(v)) if v == int(v) else f'{v:.4g}'
lines, stats = [], {}
for c in FIELDS:
    s = num[c]; x = s.dropna()
    p5, p25, p50, p75, p95 = x.quantile([.05, .25, .5, .75, .95]).tolist()
    zr, nr = (float((x == 0).mean()) if len(x) else 0.0), float(s.isna().mean())
    stats[c] = dict(p5=p5, p25=p25, p50=p50, p75=p75, p95=p95, min=float(x.min()), max=float(x.max()),
                    zero_rate=round(zr, 3), empty_rate=round(nr, 3))
    line  = f'- {c}: typical {fmt(p25)} to {fmt(p75)}, middle {fmt(p50)}, range {fmt(p5)} to {fmt(p95)} (max {fmt(x.max())})'
    extra = ([f'0 in {zr:.0%}'] if zr >= .01 else []) + ([f'NA in {nr:.0%}'] if nr >= .01 else [])
    lines.append(line + ('; ' + ', '.join(extra) if extra else ''))
ATTACK_STATS = '\n'.join(lines)
print('\n===== 5. Section 2 text (attack statistics) =====\n' + ATTACK_STATS)

# ---------- 6. Shape facts used in the prompt prose ----------
print('\n===== 6. Shape facts =====')
print(f'  single-packet flows : {rate(pk == 1):.1%}')
print(f'  port_a              : {att["port_a"].value_counts().to_dict()}')
print(f'  port_b ({att["port_b"].nunique()} unique): {sorted(att["port_b"].unique().tolist())}')
print(f'  flows per timeframe : {att["timeframe_id"].value_counts().sort_index().to_dict()}')

# ---------- 7. Section 4: fields that change from "you choose" to fixed ----------
if os.path.exists(PROMPT):
    prompt = open(PROMPT).read()
    sec4 = prompt.split('4. THE 113 FIELDS')[1].split('5. RULES FOR EMPTY')[0]
    print('\n===== 7. Section 4: "you choose" in Setting A but constant in attack =====')
    for f, _, rule in re.findall(r'^\d+\. (\w+) \(([^)]*)\) - (.*)$', sec4, re.M):
        if rule.startswith('you choose') and f in pinned:
            print(f'  {f:34s} you choose -> always {pinned[f]}')

# ---------- 8. Save ----------
out = {'source': 'train.csv attack rows only', 'n_rows': len(att),
       'pinned': pinned, 'always_empty': always_empty,
       'derived': R['derived'], 'derived_match_rate': deriv_ok,
       'empty_rules': empty_rules, 'attack_stats': stats,
       'id_formats': {'port_a': {str(k): int(v) for k, v in att['port_a'].value_counts().items()},
                      'port_b': sorted(int(p) for p in att['port_b'].unique()),
                      'timeframes': sorted(int(t) for t in att['timeframe_id'].unique())}}
json.dump(out, open(os.path.join(OUTDIR, 'rules_attack.json'), 'w'), indent=1, default=str)
open(os.path.join(OUTDIR, 'attack_stats_section.txt'), 'w').write(ATTACK_STATS)
print(f'\nSaved: {OUTDIR}/rules_attack.json, {OUTDIR}/attack_stats_section.txt')